In [2]:
# ==========================================
# DIGITAL BANKING FRAUD DETECTION SYSTEM
# Group 4 — Final Version
# Model: Gradient Boosting Classifier
# Features: 10 (after importance filtering)
# ==========================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

In [3]:
# ==========================================
# 1️⃣ LOAD DATA
# ==========================================

df = pd.read_csv("../data/PS_20174392719_1491204439457_log.csv")
print("✅ Original shape:", df.shape)


# ==========================================
# 2️⃣ FILTER RELEVANT TRANSACTION TYPES
# Only CASH_OUT and TRANSFER contain fraud
# ==========================================

df = df[df['type'].isin(['CASH_OUT', 'TRANSFER'])]
print("✅ Filtered shape:", df.shape)
print(f"   Fraud count: {df['isFraud'].sum():,}")
print(f"   Fraud rate:  {df['isFraud'].mean()*100:.3f}%")

✅ Original shape: (6362620, 11)
✅ Filtered shape: (2770409, 11)
   Fraud count: 8,213
   Fraud rate:  0.296%


In [4]:
# ==========================================
# 3️⃣ ENCODE TRANSACTION TYPE
# ==========================================

le = LabelEncoder()
df['type_encoded'] = le.fit_transform(df['type'])


# ==========================================
# 4️⃣ DERIVED FEATURES
# All built from pre-transaction columns only
# (no newbalanceOrig or newbalanceDest — post-transaction leak)
# ==========================================

# What % of sender's balance is being moved — capped at 10 to prevent explosion
df['amount_ratio'] = (df['amount'] / (df['oldbalanceOrg'] + 1)).clip(upper=10)

# Was receiver's account empty before this transaction
df['dest_was_empty'] = (df['oldbalanceDest'] == 0).astype(int)

# What hour did this transaction happen
df['hour_of_day'] = df['step'] % 24

In [5]:
# ==========================================
# 5️⃣ SPLIT BY SENDER (Leakage-Free)
# Every sender appears in ONLY train or test
# This prevents the model memorizing sender patterns
# ==========================================

all_senders = df['nameOrig'].unique()
train_senders, test_senders = train_test_split(
    all_senders,
    test_size=0.2,
    random_state=42
)

train_df = df[df['nameOrig'].isin(train_senders)].copy()
test_df  = df[df['nameOrig'].isin(test_senders)].copy()

print(f"\n✅ Train size: {len(train_df):,}")
print(f"✅ Test size:  {len(test_df):,}")

# Verify no leakage
overlap = set(train_df['nameOrig']) & set(test_df['nameOrig'])
print(f"   Sender overlap: {len(overlap)} — {'✅ No leakage!' if len(overlap)==0 else '❌ LEAKAGE!'}")


✅ Train size: 2,216,303
✅ Test size:  554,106
   Sender overlap: 0 — ✅ No leakage!


In [6]:
# ==========================================
# 6️⃣ AGGREGATION FEATURES
# Computed on TRAIN only, then merged into both
# Prevents test information leaking into training stats
# ==========================================

# Sender behavioral stats
sender_stats = train_df.groupby('nameOrig')['amount'].agg(
    sender_avg_amount='mean'
).reset_index()

# Merge into both sets
train_df = train_df.merge(sender_stats, on='nameOrig', how='left')
test_df  = test_df.merge(sender_stats,  on='nameOrig', how='left')

# Fill NaN for senders unseen in train
train_df.fillna(0, inplace=True)
test_df.fillna(0, inplace=True)

print("✅ Aggregation features created")

✅ Aggregation features created


In [7]:
# ==========================================
# 7️⃣ UNIQUE FEATURES (3 Custom Signals)
# These encode specific fraud mechanics in
# CASH_OUT and TRANSFER transactions
# ==========================================

# 🌟 #1 — Drain Score
# How much of sender's balance was drained in this transaction
# Score near 1 = account almost completely emptied in one shot
train_df['drain_score'] = train_df['amount'] / (train_df['oldbalanceOrg'] + train_df['amount'] + 1)
test_df['drain_score']  = test_df['amount']  / (test_df['oldbalanceOrg']  + test_df['amount']  + 1)

# 🌟 #2 — Both Accounts Suspicious
# Sender's balance <= amount AND receiver was empty before
# Both accounts zeroed = classic fraud chain pattern
train_df['both_accounts_suspicious'] = (
    (train_df['oldbalanceOrg'] <= train_df['amount']) &
    (train_df['oldbalanceDest'] == 0)
).astype(int)
test_df['both_accounts_suspicious'] = (
    (test_df['oldbalanceOrg'] <= test_df['amount']) &
    (test_df['oldbalanceDest'] == 0)
).astype(int)

print("✅ Unique features created")

# Verify separation (fraud vs non-fraud)
check = ['drain_score', 'both_accounts_suspicious']
print("\n=== Unique Feature Separation Check ===")
print(train_df.groupby('isFraud')[check].mean().round(4))

✅ Unique features created

=== Unique Feature Separation Check ===
         drain_score  both_accounts_suspicious
isFraud                                       
0             0.8679                     0.116
1             0.4984                     0.638


In [8]:
# ==========================================
# 8️⃣ FINAL FEATURE LIST (10 Features)
# Reduced from 15 — dropped 5 with < 1% importance
# ==========================================

features = [
    # Core (3)
    'type_encoded',
    'amount',
    'oldbalanceOrg',

    # Derived (3)
    'amount_ratio',
    'dest_was_empty',
    'hour_of_day',

    # Aggregation (1)
    'sender_avg_amount',

    # Core balance (1)
    'oldbalanceDest',

    # 🌟 Unique (2)
    'drain_score',
    'both_accounts_suspicious',
]

X_train = train_df[features]
y_train = train_df['isFraud']

X_test  = test_df[features]
y_test  = test_df['isFraud']

print(f"\n✅ Final feature count: {len(features)}")
print(f"   NaN in X_train: {X_train.isnull().sum().sum()}")
print(f"   NaN in X_test:  {X_test.isnull().sum().sum()}")


✅ Final feature count: 10
   NaN in X_train: 0
   NaN in X_test:  0


In [9]:
# ==========================================
# 9️⃣ TRAIN GRADIENT BOOSTING
# Using sample_weight to handle class imbalance
# (sklearn GradientBoosting does not support scale_pos_weight)
# ==========================================

# Calculate class imbalance ratio
fraud_count     = y_train.sum()
non_fraud_count = (y_train == 0).sum()
scale           = non_fraud_count / fraud_count

# Give fraud rows higher weight
sample_weights = np.where(y_train == 1, scale, 1)

print(f"\n✅ Class imbalance ratio: {scale:.2f}")
print(f"   Fraud weight: {scale:.0f}x more important than non-fraud")

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    min_samples_leaf=100,
    subsample=0.8,
    max_features='sqrt',
    random_state=42,
    verbose=1
)

print("\n⏳ Training model...")
gb_model.fit(X_train, y_train, sample_weight=sample_weights)
print("✅ Training complete!")


✅ Class imbalance ratio: 340.07
   Fraud weight: 340x more important than non-fraud

⏳ Training model...
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           1.2896           0.0969            9.14m
         2           1.2084           0.0812            6.94m
         3           1.1283           0.0806            6.92m
         4           1.0547           0.0724            6.82m
         5           0.9878           0.0675            6.38m
         6           0.9285           0.0596            5.89m
         7           0.8718           0.0572            5.77m
         8           0.8193           0.0507            5.48m
         9           0.7787           0.0411            5.37m
        10           0.7333           0.0438            5.16m
        20           0.4197           0.0258            4.84m
        30           0.2547           0.0151            4.60m
        40           0.1596           0.0087            3.78m
        50           0.10

In [10]:
# ==========================================
# 🔟 PREDICTIONS
# ==========================================

y_pred  = gb_model.predict(X_test)
y_proba = gb_model.predict_proba(X_test)[:, 1]


# ==========================================
# 1️⃣1️⃣ EVALUATION
# ==========================================

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("\n" + "="*50)
print("   CONFUSION MATRIX")
print("="*50)
print(f"\n                 Predicted LEGIT  Predicted FRAUD")
print(f"Actual LEGIT     {cm[0][0]:>13,}  {cm[0][1]:>14,}")
print(f"Actual FRAUD     {cm[1][0]:>13,}  {cm[1][1]:>14,}")

print("\n" + "="*50)
print("   CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

print("\n" + "="*50)
print("   KEY FRAUD METRICS")
print("="*50)
print(f"Total fraud in test:          {tp+fn:,}")
print(f"Fraud caught (True Positive): {tp:,}")
print(f"Fraud missed (False Negative):{fn:,}")
print(f"False alarms (False Positive):{fp:,}")
print(f"\nFraud Precision: {tp/(tp+fp)*100:.1f}%")
print(f"Fraud Recall:    {tp/(tp+fn)*100:.1f}%")


   CONFUSION MATRIX

                 Predicted LEGIT  Predicted FRAUD
Actual LEGIT           552,113             278
Actual FRAUD                 5           1,710

   CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    552391
           1       0.86      1.00      0.92      1715

    accuracy                           1.00    554106
   macro avg       0.93      1.00      0.96    554106
weighted avg       1.00      1.00      1.00    554106

ROC-AUC Score: 0.9997

   KEY FRAUD METRICS
Total fraud in test:          1,715
Fraud caught (True Positive): 1,710
Fraud missed (False Negative):5
False alarms (False Positive):278

Fraud Precision: 86.0%
Fraud Recall:    99.7%


In [11]:
# ==========================================
# 1️⃣2️⃣ FEATURE IMPORTANCE
# ==========================================

importance_df = pd.DataFrame({
    'feature':    features,
    'importance': gb_model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

importance_df['importance_pct'] = (importance_df['importance'] * 100).round(2)
importance_df['rank'] = range(1, len(importance_df) + 1)

print("\n" + "="*50)
print("   FEATURE IMPORTANCE")
print("="*50)
print(importance_df[['rank','feature','importance_pct']].to_string(index=False))


# ==========================================
# 1️⃣3️⃣ 3-LEVEL RISK CLASSIFICATION
# ==========================================

def classify_transaction(prob):
    if prob >= 0.55:
        return "FRAUD ALERT 🚨"
    elif prob >= 0.15:
        return "SUSPICIOUS ⚠️"
    else:
        return "LEGIT ✅"

predicted_labels = [classify_transaction(p) for p in y_proba]

print("\n=== Risk Label Distribution ===")
print(pd.Series(predicted_labels).value_counts())


# ==========================================
# 1️⃣4️⃣ COMPARE WITH BANK RULE
# ==========================================

bank_cm = confusion_matrix(y_test, test_df['isFlaggedFraud'])
bank_tn, bank_fp, bank_fn, bank_tp = bank_cm.ravel()

print("\n" + "="*50)
print("   HEAD TO HEAD: OUR MODEL vs BANK RULE")
print("="*50)
print(f"{'Metric':<25} {'Bank Rule':>12} {'Our Model':>12}")
print("-"*50)
print(f"{'Fraud Caught':<25} {bank_tp:>12,} {tp:>12,}")
print(f"{'Fraud Missed':<25} {bank_fn:>12,} {fn:>12,}")
print(f"{'False Alarms':<25} {bank_fp:>12,} {fp:>12,}")
print(f"{'Recall':<25} {bank_tp/(bank_tp+bank_fn)*100:>11.1f}% {tp/(tp+fn)*100:>11.1f}%")
print(f"{'Precision':<25} {(bank_tp/(bank_tp+bank_fp)*100) if (bank_tp+bank_fp)>0 else 0:>11.1f}% {tp/(tp+fp)*100:>11.1f}%")


   FEATURE IMPORTANCE
 rank                  feature  importance_pct
    1              drain_score           52.25
    2             amount_ratio           29.62
    3            oldbalanceOrg            4.91
    4 both_accounts_suspicious            4.67
    5           oldbalanceDest            3.08
    6        sender_avg_amount            1.76
    7                   amount            1.47
    8              hour_of_day            1.14
    9           dest_was_empty            0.67
   10             type_encoded            0.43

=== Risk Label Distribution ===
LEGIT ✅          551775
FRAUD ALERT 🚨      1951
SUSPICIOUS ⚠️       380
Name: count, dtype: int64

   HEAD TO HEAD: OUR MODEL vs BANK RULE
Metric                       Bank Rule    Our Model
--------------------------------------------------
Fraud Caught                         5        1,710
Fraud Missed                     1,710            5
False Alarms                         0          278
Recall                       

In [12]:
# ==========================================
# 1️⃣7️⃣ ISOLATION FOREST
# Unsupervised anomaly detection
# Trained on LEGIT transactions only —
# learns what normal looks like, flags anomalies
# ==========================================

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler

# Train on LEGIT transactions only
X_train_legit = X_train[y_train == 0]
print(f"Training Isolation Forest on {len(X_train_legit):,} legitimate transactions...")

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.003,  # matches our ~0.3% fraud rate
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train_legit)
print("✅ Isolation Forest trained!")

# Get anomaly scores for test set
# More negative = more anomalous
iso_scores_raw = iso_forest.decision_function(X_test)

# Flip and normalize to 0-1 scale
# High score = high anomaly = high fraud risk
scaler = MinMaxScaler()
iso_scores = scaler.fit_transform(-iso_scores_raw.reshape(-1, 1)).flatten()

print(f"\n✅ Anomaly scores computed")
print(f"   Score range: {iso_scores.min():.3f} to {iso_scores.max():.3f}")
print("\n=== Isolation Forest Score: Fraud vs Non-Fraud ===")
iso_check = pd.DataFrame({'iso_score': iso_scores, 'isFraud': y_test.values})
print(iso_check.groupby('isFraud')['iso_score'].mean().round(4))


Training Isolation Forest on 2,209,805 legitimate transactions...
✅ Isolation Forest trained!

✅ Anomaly scores computed
   Score range: 0.000 to 1.000

=== Isolation Forest Score: Fraud vs Non-Fraud ===
isFraud
0    0.2099
1    0.6923
Name: iso_score, dtype: float64


In [13]:
# ==========================================
# 1️⃣8️⃣ COMBINE MODELS — WEIGHTED SCORE
# 70% Gradient Boosting + 30% Isolation Forest
# GB is stronger so gets higher weight
# IF catches anomalies GB might miss
# ==========================================

GB_WEIGHT  = 0.70
IF_WEIGHT  = 0.30

# Combined fraud probability
combined_score = (GB_WEIGHT * y_proba) + (IF_WEIGHT * iso_scores)

print(f"✅ Combined score = {GB_WEIGHT*100:.0f}% GB + {IF_WEIGHT*100:.0f}% IF")
print(f"   Score range: {combined_score.min():.3f} to {combined_score.max():.3f}")

# 3-Level classification on combined score
def classify_combined(score):
    if score >= 0.55:
        return "FRAUD ALERT 🚨"
    elif score >= 0.15:
        return "SUSPICIOUS ⚠️"
    else:
        return "LEGIT ✅"

combined_labels = [classify_combined(s) for s in combined_score]

# Binary prediction from combined score
y_pred_combined = (combined_score >= 0.55).astype(int)

# Evaluation
cm_combined = confusion_matrix(y_test, y_pred_combined)
tn_c, fp_c, fn_c, tp_c = cm_combined.ravel()

print("\n" + "="*50)
print("   COMBINED MODEL CONFUSION MATRIX")
print("="*50)
print(f"\n                 Predicted LEGIT  Predicted FRAUD")
print(f"Actual LEGIT     {cm_combined[0][0]:>13,}  {cm_combined[0][1]:>14,}")
print(f"Actual FRAUD     {cm_combined[1][0]:>13,}  {cm_combined[1][1]:>14,}")

print("\n" + "="*50)
print("   COMBINED MODEL REPORT")
print("="*50)
print(classification_report(y_test, y_pred_combined))
print(f"ROC-AUC Score: {roc_auc_score(y_test, combined_score):.4f}")

print("\n=== Risk Label Distribution ===")
print(pd.Series(combined_labels).value_counts())


✅ Combined score = 70% GB + 30% IF
   Score range: 0.008 to 0.997

   COMBINED MODEL CONFUSION MATRIX

                 Predicted LEGIT  Predicted FRAUD
Actual LEGIT           552,197             194
Actual FRAUD                 6           1,709

   COMBINED MODEL REPORT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    552391
           1       0.90      1.00      0.94      1715

    accuracy                           1.00    554106
   macro avg       0.95      1.00      0.97    554106
weighted avg       1.00      1.00      1.00    554106

ROC-AUC Score: 0.9993

=== Risk Label Distribution ===
LEGIT ✅          487266
SUSPICIOUS ⚠️     64937
FRAUD ALERT 🚨      1903
Name: count, dtype: int64


In [14]:
# ==========================================
# 1️⃣9️⃣ HEAD TO HEAD — ALL 3 SYSTEMS
# Bank Rule vs GB Only vs GB + IF Combined
# ==========================================

print("\n" + "="*60)
print("   FULL COMPARISON: BANK RULE vs GB vs GB + IF")
print("="*60)
print(f"{'Metric':<25} {'Bank Rule':>12} {'GB Only':>10} {'GB + IF':>10}")
print("-"*60)
print(f"{'Fraud Caught':<25} {bank_tp:>12,} {tp:>10,} {tp_c:>10,}")
print(f"{'Fraud Missed':<25} {bank_fn:>12,} {fn:>10,} {fn_c:>10,}")
print(f"{'False Alarms':<25} {bank_fp:>12,} {fp:>10,} {fp_c:>10,}")
print(f"{'Recall':<25} {bank_tp/(bank_tp+bank_fn)*100:>11.1f}% {tp/(tp+fn)*100:>9.1f}% {tp_c/(tp_c+fn_c)*100:>9.1f}%")
print(f"{'Precision':<25} {(bank_tp/(bank_tp+bank_fp)*100) if (bank_tp+bank_fp)>0 else 0:>11.1f}% {tp/(tp+fp)*100:>9.1f}% {tp_c/(tp_c+fp_c)*100:>9.1f}%")
print(f"{'ROC-AUC':<25} {'N/A':>12} {roc_auc_score(y_test, y_proba):>10.4f} {roc_auc_score(y_test, combined_score):>10.4f}")



   FULL COMPARISON: BANK RULE vs GB vs GB + IF
Metric                       Bank Rule    GB Only    GB + IF
------------------------------------------------------------
Fraud Caught                         5      1,710      1,709
Fraud Missed                     1,710          5          6
False Alarms                         0        278        194
Recall                            0.3%      99.7%      99.7%
Precision                       100.0%      86.0%      89.8%
ROC-AUC                            N/A     0.9997     0.9993


In [15]:
# ==========================================
# 2️⃣0️⃣ SAMPLE PREDICTIONS — COMBINED
# ==========================================

results_combined = pd.DataFrame({
    "isFraud":           y_test.values,
    "gb_prob":           y_proba.round(3),
    "if_score":          iso_scores.round(3),
    "combined_score":    combined_score.round(3),
    "predicted_label":   combined_labels,
    "isFlaggedFraud":    test_df['isFlaggedFraud'].values
})

print("Sample Predictions (Combined Model):")
print(results_combined.head(10))

# Show some caught fraud cases
print("\n=== Sample FRAUD Cases Caught ===")
print(results_combined[
    (results_combined['isFraud'] == 1) &
    (results_combined['predicted_label'] == 'FRAUD ALERT 🚨')
].head(5))


Sample Predictions (Combined Model):
   isFraud  gb_prob  if_score  combined_score predicted_label  isFlaggedFraud
0        1    0.962     0.430           0.802   FRAUD ALERT 🚨               0
1        0    0.010     0.355           0.113         LEGIT ✅               0
2        0    0.062     0.594           0.222   SUSPICIOUS ⚠️               0
3        0    0.034     0.488           0.170   SUSPICIOUS ⚠️               0
4        0    0.048     0.423           0.161   SUSPICIOUS ⚠️               0
5        0    0.016     0.377           0.125         LEGIT ✅               0
6        0    0.016     0.377           0.124         LEGIT ✅               0
7        0    0.016     0.388           0.128         LEGIT ✅               0
8        0    0.018     0.538           0.174   SUSPICIOUS ⚠️               0
9        0    0.017     0.387           0.128         LEGIT ✅               0

=== Sample FRAUD Cases Caught ===
     isFraud  gb_prob  if_score  combined_score predicted_label  \
0  

In [17]:
# ==========================================
# 2️⃣1️⃣ SAVE BOTH MODELS & FEATURES
# ==========================================

os.makedirs("../models", exist_ok=True)

# Save Gradient Boosting model
joblib.dump(gb_model,   "../models/gradient_boosting_fraud_model.pkl")

# Save Isolation Forest model
joblib.dump(iso_forest, "../models/isolation_forest_model.pkl")

# Save MinMaxScaler (needed to normalize IF scores at inference)
joblib.dump(scaler,     "../models/iso_scaler.pkl")

# Save feature list
joblib.dump(features,   "../models/feature_list.pkl")

# Save weights
joblib.dump({'gb': GB_WEIGHT, 'if': IF_WEIGHT}, "../models/model_weights.pkl")

print("✅ Gradient Boosting saved: ../models/gradient_boosting_fraud_model.pkl")
print("✅ Isolation Forest saved:  ../models/isolation_forest_model.pkl")
print("✅ IF Scaler saved:         ../models/iso_scaler.pkl")
print("✅ Feature list saved:      ../models/feature_list.pkl")
print("✅ Model weights saved:     ../models/model_weights.pkl")
print("\n🎉 Done — Group 4 Fraud Detection System (GB + IF) Complete!")


✅ Gradient Boosting saved: ../models/gradient_boosting_fraud_model.pkl
✅ Isolation Forest saved:  ../models/isolation_forest_model.pkl
✅ IF Scaler saved:         ../models/iso_scaler.pkl
✅ Feature list saved:      ../models/feature_list.pkl
✅ Model weights saved:     ../models/model_weights.pkl

🎉 Done — Group 4 Fraud Detection System (GB + IF) Complete!
